# Machine Learning

**Case study:** Customer churn prediction and segmentation. Synthetic data only. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data") if Path("data/customers_synthetic.csv").exists() else Path(".")
customers = pd.read_csv(DATA / "customers_synthetic.csv")
orders = pd.read_csv(DATA / "orders_synthetic.csv", parse_dates=["order_date"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head())

## Data quality and exploratory checks

In [ ]:
assert customers.customer_id.is_unique
assert orders.order_id.is_unique
assert customers.churned.isin([0,1]).all()
print(customers.isna().sum().to_string())
print("Churn rate:", round(customers.churned.mean(),3))
customers.groupby("plan").churned.agg(["mean","count"]).plot(kind="bar",subplots=True,figsize=(9,3),title=["Churn rate","Customers"]); plt.tight_layout(); plt.show()

## Supervised classification: baseline and leakage-safe preprocessing

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
features=["age","tenure_months","monthly_spend","support_tickets","discount_pct","engagement_score","region","plan"]
X=customers[features]; y=customers.churned
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
num=X.select_dtypes(include="number").columns.tolist(); cat=[c for c in X if c not in num]
pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),("cat",OneHotEncoder(handle_unknown="ignore"),cat)])
model=Pipeline([("pre",pre),("clf",LogisticRegression(max_iter=1000))])
model.fit(X_train,y_train)
pred=model.predict(X_test); proba=model.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred,zero_division=0))
print("ROC AUC:",round(roc_auc_score(y_test,proba),3))
print("Dummy accuracy:",round(DummyClassifier(strategy="most_frequent").fit(X_train,y_train).score(X_test,y_test),3))
ConfusionMatrixDisplay.from_predictions(y_test,pred);plt.show()

## Cross-validation and random forest comparison

In [ ]:
from sklearn.ensemble import RandomForestClassifier
cv=StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
print("Logistic CV ROC AUC:",cross_val_score(model,X_train,y_train,cv=cv,scoring="roc_auc").mean().round(3))
forest=Pipeline([("pre",pre),("clf",RandomForestClassifier(n_estimators=100,min_samples_leaf=4,random_state=42))])
forest.fit(X_train,y_train)
print("Forest test ROC AUC:",round(roc_auc_score(y_test,forest.predict_proba(X_test)[:,1]),3))

## Unsupervised learning: K-means segmentation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
segment_cols=["monthly_spend","tenure_months","engagement_score"]
scaled=StandardScaler().fit_transform(customers[segment_cols])
labels=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(scaled)
print("Silhouette score:",round(silhouette_score(scaled,labels),3))
display(customers.assign(segment=labels).groupby("segment")[segment_cols].mean().round(1))

## Reflection

1. What was your baseline? 2. What could go wrong in deployment? 3. Which results require human review?

Complete `Practice_Questions.md` before reading `Solutions.md`.